In [1]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from Parsing import split_resume_into_sections
import os, glob

folder = "../../Training_Data/Resume/"
pdf_paths = sorted(glob.glob(os.path.join(folder, "*.pdf")))
print(f"Found {len(pdf_paths)} PDF(s)\n")

titles = [] # 你 parse 完的標題 list
for path in pdf_paths:
    name = os.path.basename(path)
    print(f"\n########## {name} ##########")
    try:
        sections = split_resume_into_sections(path)
        titles.extend(sec["heading"] for sec in sections)
        # for i, sec in enumerate(sections):
        #     print(f"\n===== Section {i}: {sec['heading']!r} =====")
        #     print(sec["content"][:150])
    except Exception as e:
        print(f"✗ FAILED: {e}")


  

embedding_model = SentenceTransformer("BAAI/bge-base-en-v1.5")


umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine")
hdbscan_model = HDBSCAN(min_cluster_size=5, metric="euclidean",
                        cluster_selection_method="eom", prediction_data=True)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    min_topic_size=5,
)

topics, probs = topic_model.fit_transform(titles)
topic_model.get_topic_info()

c:\Users\erank\anaconda3\envs\resumechecker\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Found 12 PDF(s)


########## Bharath_Ganesh_Lead.pdf ##########

########## BhaveshSonje(Resume).pdf ##########

########## Eranki Vasistha_SWE.pdf ##########

########## Hardik_SWE_v3.pdf ##########

########## Harisankar_Kartha_Resume.pdf ##########

########## Mantri_Aditya_SDE.pdf ##########

########## Resume.pdf ##########

########## Resume_Sylvey Lin_SDE.pdf ##########

########## Resume_Yu-Wen 'Yvonne' Yang.pdf ##########

########## Resume_grad.pdf ##########

########## YiHan_Huang_resume.pdf ##########

########## resumeangad.pdf ##########


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7297.06it/s]


,Topic,Count,Name,Representation,Representative_Docs
0,0,16,0_header_kills_xperience_rojects,"[header, kills, xperience, rojects, ork, ducat...","[HEADER, HEADER, HEADER]"
1,1,16,1_skills_technical_activities_hyderabad,"[skills, technical, activities, hyderabad, ind...","[TECHNICAL SKILLS, TECHNICAL SKILLS, Technical..."
2,2,13,2_experience_work_experiences_vasistha,"[experience, work, experiences, vasistha, rese...","[Experience, EXPERIENCE, Experience]"
3,3,11,3_education_background__,"[education, background, , , , , , , , ]","[Education, EDUCATION, EDUCATION]"
4,4,11,4_projects_selected_summary_,"[projects, selected, summary, , , , , , , ]","[PROJECTS, PROJECTS, Projects]"


In [2]:
import pandas as pd

records = []
for path in pdf_paths:
    resume_id = os.path.splitext(os.path.basename(path))[0]   # filename = resume_id
    try:
        for sec in split_resume_into_sections(path):
            records.append({
                "resume_id": resume_id,
                "heading":   sec["heading"],
                "content":   sec["content"],
            })
    except Exception as e:
        print(f"✗ {resume_id} FAILED: {e}")

sections_df = pd.DataFrame(records)
titles = sections_df["heading"].tolist()      # <- feeds BERTopic, same as before
print(sections_df.shape)
sections_df.head()

(67, 3)


,resume_id,heading,content
0,Bharath_Ganesh_Lead,HEADER,"Bharath Ganesh\n| | | Seattle,WA|bharath110996..."
1,Bharath_Ganesh_Lead,S UMMARY,Lead Software Engineer with 6+ years deliverin...
2,Bharath_Ganesh_Lead,E DUCATION,Aug2024–May2026 UniversityofIllinoisUrbana-Cha...
3,Bharath_Ganesh_Lead,W E ORK XPERIENCE,"| Urbana,IL Jun2025–May2026 WebsiteAdministrat..."
4,Bharath_Ganesh_Lead,S KILLS,"Java,Go,Python,JavaScript,TypeScript,SQL,Bash ..."


In [4]:
# With only ~10 resumes most headings land in topic -1 (outliers).
# Reassign outliers to the nearest real topic so they can be merged.
# topics = topic_model.reduce_outliers(titles, topics, strategy="embeddings")
sections_df["topic"] = topics

# canonical section name = the most common heading text within each topic
canonical = (sections_df.groupby("topic")["heading"]
             .agg(lambda s: s.value_counts().index[0]))
sections_df["section"] = sections_df["topic"].map(canonical)

# sanity-check the clustering before trusting it
sections_df.groupby("section")["heading"].unique()

section
EDUCATION          [EDUCATION, Education, EDUCATION BACKGROUND]
EXPERIENCE    [WORK EXPERIENCE, Vasistha Eranki, EXPERIENCE,...
HEADER        [HEADER, S UMMARY, E DUCATION, W E ORK XPERIEN...
PROJECTS       [PROJECTS, Projects, SELECTED PROJECTS, Summary]
SKILLS        [SKILLS, TECHNICAL SKILLS, Full Stack Develope...
Name: heading, dtype: object

In [5]:
# concatenate content of all sections that share a canonical name, per resume
merged = (sections_df
          .groupby(["resume_id", "section"])["content"]
          .apply(lambda c: "\n".join(x for x in c if x))
          .reset_index())

# wide format: one row per resume, one column per canonical section
resume_parsed = (merged.pivot(index="resume_id", columns="section", values="content")
                       .fillna("")
                       .reset_index())
resume_parsed.columns.name = None
resume_parsed.to_csv("resume_parsed.csv", index=False)
print(resume_parsed.shape)
resume_parsed.head()

(12, 6)


,resume_id,EDUCATION,EXPERIENCE,HEADER,PROJECTS,SKILLS
0,Bharath_Ganesh_Lead,,,"Bharath Ganesh\n| | | Seattle,WA|bharath110996...",,
1,BhaveshSonje(Resume),"University of Illinois, Urbana Champaign | Cha...",Apexon (Goldman Sachs Portfolio Company) Decem...,BHAVESH SONJE\n+1 (447) 902-4869 | bsonje2@ill...,CompetIQ - Autonomous Competitive Intelligence...,"Product : Product Strategy, 0 to 1 Delivery, R..."
2,Eranki Vasistha_SWE,University of Illinois Urbana-Champaign Aug 20...,"Urbana, IL | 217-255-3348 | veranki2@illinois....",,Biomedical LLM Clarification-Seeking Pipeline ...,"• AI/ML : PyTorch, Transformers, Deep Learning..."
3,Hardik_SWE_v3,University of Illinois Urbana-Champaign Urbana...,Teaching Assistant – BADM 579 and BADM 550 Aug...,Hardik Lad\n217-693-1462 | hardiklad1@hotmail....,RainStorm — Distributed Stream Processing Engi...,"Languages : Python, JavaScript (ES6+), SQL, Ba..."
4,Harisankar_Kartha_Resume,"University of Illinois, Urbana - Champaign Aug...",Infosys Limited Dec 2021 – Jul 2024\nSr. Syste...,"Harisankar Kartha\nUrbana, IL\n(+1) 217-693-12...",Semantic Drift Detection in AI Terminology | P...,"Python, SQL, Scala, Java, Go Languages:\nApach..."


In [7]:
resume = pd.read_csv("resume_parsed.csv")
jd     = pd.read_csv("job_descriptions_tech_pref.csv")   # your both-non-empty filtered file

# sample 2,200 job descriptions at random (reproducible)
jd = jd.sample(n=2200, random_state=0).reset_index(drop=True)

# prefix columns so resume vs JD fields never collide after the join
resume = resume.add_prefix("resume_")     # -> resume_resume_id, resume_Education, ...
jd     = jd.add_prefix("jd_")             # -> jd_job_id, jd_job_title, ...

pairs = resume.merge(jd, how="cross")
print(f"{len(resume)} resumes × {len(jd)} jobs = {len(pairs):,} pairs")
pairs.to_csv("resume_jd_pairs.csv", index=False)

12 resumes × 2200 jobs = 26,400 pairs
